# Auto Merging Retriever for Advanced RAG

This notebook explains and demonstrates the Auto Merging Retriever technique using LlamaIndex, a retrieval strategy that searches small precise chunks but automatically expands them back into larger context when enough of them point to the same source passage.

## What problem does this solve?

The Sentence Window notebook in this series already showed one way to balance precision and context, by searching a single sentence and then swapping in a wider window of neighboring sentences after retrieval. Auto Merging Retrieval solves a very similar underlying problem, but with a tree shaped chunking structure instead of a fixed size window.

The core tension is the same one that shows up throughout RAG chunking decisions.

1. Small chunks give sharp, focused embeddings, so similarity search is more likely to find the exact passage that answers a question.
2. Small chunks in isolation can be missing context that a slightly larger surrounding passage would have provided.
3. Simply making every chunk larger trades away that search precision, since a big chunk mixes several ideas into one embedding.

Auto Merging Retrieval resolves this by first breaking each document into a hierarchy of chunks at several sizes, for example large parent chunks, medium child chunks, and small leaf chunks, where every leaf chunk knows which parent chunk it belongs to. Only the smallest leaf chunks are embedded and searched, which keeps retrieval precise. After the search returns its results, the retriever checks whether several leaf chunks that were retrieved all happen to belong to the same parent chunk. If enough of them do, it assumes the parent chunk as a whole is relevant, and it automatically replaces those scattered leaf chunks with the single larger parent chunk before anything is sent to the language model. This gives the language model a more complete, coherent piece of context exactly when the evidence suggests it is needed, while leaving isolated single leaf matches as they are the rest of the time.

## What this notebook covers

1. Importing the classes needed to build a hierarchical, auto merging retrieval pipeline.
2. Building a hierarchical node parser that splits documents into parent, child, and leaf chunks.
3. Loading a source document from the web.
4. Converting that document into the full hierarchy of nodes, and isolating just the leaf nodes for indexing.
5. Configuring the embedding model used for similarity search.
6. Building a storage context that keeps every node in the hierarchy available, and a vector index built only from the leaf nodes.
7. Building the Auto Merging Retriever itself, which wraps a normal similarity retriever with the automatic merging logic.
8. Configuring the language model and assembling a full query engine around the retriever.
9. Asking a question end to end and inspecting the generated answer.
10. A diagram summarizing the complete pipeline.
11. A conclusion tying the technique back to the observations made in this notebook.


## Section 1. Importing the required classes

Every class needed to build the pipeline is imported up front. Each one is explained here so its purpose is clear before it appears in the pipeline later on, rather than being introduced all at once without context.

1. `StorageContext` is the object responsible for holding every node produced during parsing, including the larger parent and child nodes that are not directly searched, so they remain available for the retriever to pull in later.
2. `ServiceContext` bundles together model and parser settings such as the language model, the embedding model, and the node parser, so they can be applied consistently wherever they are needed. It is imported here for completeness even though this notebook wires the language model in through a separate global settings object instead.
3. `VectorStoreIndex` builds the searchable vector index, here built only from the smallest leaf nodes.
4. `HierarchicalNodeParser` is the parser that splits a document into multiple levels of chunks at once, for example large parent chunks, medium child chunks, and small leaf chunks, keeping track of which chunks are children of which.
5. `get_leaf_nodes` is a helper function that walks the full hierarchy produced by the parser and returns only the smallest, most granular nodes, which is exactly the set of nodes that should be embedded and searched.
6. `AutoMergingRetriever` is the retriever that performs the automatic merging logic described in the overview, replacing groups of retrieved leaf nodes with their shared parent node whenever enough of them agree.
7. `RetrieverQueryEngine` wraps any retriever together with a language model, turning it into a complete query engine that can accept a natural language question and return a natural language answer.
8. `SentenceTransformerRerank` is a reranking postprocessor that can reorder retrieved nodes by relevance using a dedicated cross encoder model. It is imported here for reference as a possible extension to this pipeline, even though this notebook does not use it directly in the final query engine.


In [ ]:
from llama_index.core import StorageContext, ServiceContext, VectorStoreIndex
from llama_index.core.node_parser import (
    HierarchicalNodeParser,
    get_leaf_nodes
)
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.postprocessor import SentenceTransformerRerank

## Section 2. Creating a hierarchical node parser

Instead of splitting a document into chunks of a single fixed size, `HierarchicalNodeParser` splits it into several nested levels of chunks at once. The `chunk_sizes` argument below defines three levels.

1. Level 1, the parent level, targets chunks of roughly 1024 tokens.
2. Level 2, the child level, targets chunks of roughly 512 tokens, where every child chunk falls inside exactly one parent chunk.
3. Level 3, the leaf level, targets chunks of roughly 128 tokens, where every leaf chunk falls inside exactly one child chunk.

Only the leaf level chunks will actually be embedded and searched later. The parent and child levels exist so that when several leaf level matches point back to the same larger chunk, that larger chunk can be reconstructed and used instead, giving the language model more complete context exactly when the evidence supports it.


In [ ]:
# Create a hierarchical text splitter
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[1024, 512, 128]   # Parent -> Child -> Leaf
)

## Section 3. Loading the source document

With the parser configured, the next step is to load the raw text that will be split and indexed. This notebook loads a blog post about LLM powered autonomous agents directly from the web, the same source document used elsewhere in this series, which makes it easy to compare retrieval strategies across notebooks using consistent source material.


In [ ]:
from llama_index.readers.web import SimpleWebPageReader

# Load content directly from a webpage
documents = SimpleWebPageReader().load_data(
    urls=["https://lilianweng.github.io/posts/2023-06-23-agent/"]
)

Checking how many documents were loaded is a quick sanity check. A single web page typically loads as a single `Document` object containing the page's full text.


In [ ]:
len(documents)

## Section 4. Converting documents into hierarchical nodes

Running the hierarchical parser on the loaded documents produces a tree structure for each document, roughly shaped like this.

```text
Document
  Parent Node (1024)
  Child Node (512)
  Leaf Node (128)
  Leaf Node (128)
```

Every node in this tree is kept, but only the leaf nodes, the smallest and most numerous ones, are intended for direct similarity search. `get_leaf_nodes` filters the full set of nodes down to just those leaf level nodes.


In [ ]:
# Split documents into hierarchical nodes
nodes = node_parser.get_nodes_from_documents(documents)

# Extract only the smallest chunks (leaf nodes)
leaf_nodes = get_leaf_nodes(nodes)

Checking how many leaf nodes were produced gives a sense of how finely the source document was ultimately broken down for search purposes.


In [ ]:
len(leaf_nodes)

Printing the first leaf node shows exactly what a single unit of searchable text looks like at the smallest level of the hierarchy, which is useful context for interpreting retrieval results later in the notebook.


In [ ]:
print(leaf_nodes[0])

## Section 5. Initializing the embedding model

An embedding model is needed to turn each leaf node's text into a vector for similarity search. This notebook uses a HuggingFace BGE embedding model, `BAAI/bge small en v1.5`, which is a model specifically trained to produce embeddings well suited for retrieval tasks.

Setting `normalize_embeddings` to true means every embedding vector produced by the model has unit length, which allows a simple dot product between two vectors to serve as their cosine similarity, the standard similarity measure used when comparing text embeddings.


In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# Name of the Hugging Face embedding model
model_name = "BAAI/bge-small-en-v1.5"

# Normalize embeddings for cosine similarity
encode_kwargs = {
    "normalize_embeddings": True
}

# Initialize the embedding model
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    # model_kwargs={"device": "cuda"},  # Uncomment to run on GPU
    encode_kwargs=encode_kwargs,
)

### Bridging LangChain and LlamaIndex

The embedding model above was created using LangChain's HuggingFace wrapper, but the rest of this notebook is built on LlamaIndex. `LangchainEmbedding` is an adapter class that wraps a LangChain embedding object so that LlamaIndex components, such as `VectorStoreIndex`, can call it as if it were a native LlamaIndex embedding model. This adapter is what makes it possible to combine tools from both frameworks in a single pipeline.


In [ ]:
from llama_index.embeddings.langchain import LangchainEmbedding

## Section 6. Creating the storage context and the vector index

Two related but distinct things are being built in this step.

1. A `StorageContext` that stores every node produced by the hierarchical parser, including the parent and child nodes, even though those larger nodes are never directly embedded or searched. They need to remain available in storage so the Auto Merging Retriever can look them up and substitute them in later, whenever enough of their child leaf nodes are retrieved together.
2. A `VectorStoreIndex` built using only the leaf nodes, since searching the smallest, most focused chunks produces more accurate similarity search results than searching the larger parent or child chunks directly.

Put simply, the storage context remembers the entire hierarchy, while the vector index only searches its bottom layer.


In [ ]:
# Create storage for all hierarchical nodes
storage_context = StorageContext.from_defaults()

# Store all nodes (parent and child nodes)
storage_context.docstore.add_documents(nodes)

# Build the vector index using the leaf nodes
index = VectorStoreIndex(
    leaf_nodes,
    storage_context=storage_context,
    embed_model=LangchainEmbedding(embedding_function)
)

### Inspecting the index object

Printing the type of the index confirms exactly what kind of LlamaIndex object was built, and listing its attributes with `dir` gives a quick look at everything the object exposes, which is a useful way to explore an unfamiliar class directly rather than only reading documentation.


In [ ]:
print(type(index))

In [ ]:
print(dir(index))

### Confirming the full hierarchy is stored

Counting the documents held in the storage context's docstore should reflect the full hierarchy, meaning parent, child, and leaf nodes all together, which is noticeably more than the leaf node count alone. This confirms the parent and child nodes really were preserved in storage even though they are absent from the searchable vector index itself.


In [ ]:
print(len(index.storage_context.docstore.docs))

Pulling out and printing the text of the first node stored in the docstore is a final concrete check that the storage context is holding real, readable node content, not just empty placeholders.


In [ ]:
first_node = next(iter(index.storage_context.docstore.docs.values()))

print(first_node.text)

## Section 7. Creating the Auto Merging Retriever

This is where the core technique from the overview is actually assembled. The process happens in two layers.

1. A normal similarity retriever is built from the vector index, configured to return the top 6 most similar leaf nodes for any given query.
2. That base retriever is wrapped inside an `AutoMergingRetriever`, along with the storage context that holds the full node hierarchy. When this wrapped retriever runs, it first performs the same similarity search as the base retriever, then checks whether enough of the returned leaf nodes share a common parent. If so, it replaces those leaf nodes with the shared parent node before returning its final results.

For example, if the search happens to retrieve three separate leaf nodes that all belong to the same parent node, the retriever recognizes this pattern and merges them.

```text
Retrieved
  Child 1
  Child 2
  Child 3
      |
      v
Merged into
  Parent Node
```

Setting `verbose=True` makes the retriever print information about any merges it performs while it runs, which is useful for understanding, during actual queries, when and why the automatic merging behavior kicked in.


In [ ]:
# Create the basic similarity retriever
base_retriever = index.as_retriever(
    similarity_top_k=6   # Retrieve top 6 leaf nodes
)

# Automatically merge nearby child nodes into their parent node
retriever = AutoMergingRetriever(
    base_retriever,
    storage_context,
    verbose=True          # Print merging information
)

## Section 8. Building the query engine

A retriever on its own only fetches relevant nodes. It does not generate a natural language answer. `RetrieverQueryEngine` is the class that closes this gap, connecting a retriever to a language model so that a full question can be answered end to end. Its workflow is as follows.

```text
User Question
      |
      v
Retriever
      |
      v
Relevant Nodes
      |
      v
LLM
      |
      v
Final Answer
```


### Configuring the language model credentials

As with the other notebooks in this series, the Gemini API key is loaded from a local `.env` file rather than typed directly into the notebook. This keeps the real credential out of the notebook file itself, which matters if the notebook is later pushed to a public repository such as GitHub.


In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Get the API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

### Configuring the language model

`Settings.llm` is LlamaIndex's global configuration object. Setting it once here means every component built afterward, including the query engine assembled below, automatically uses this Gemini model without needing to pass it in explicitly every time. This notebook uses `gemini 2.5 flash` as the language model that will generate the final answer.


In [ ]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core import Settings

# Configure the Gemini LLM
Settings.llm = GoogleGenAI(
    model="gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
)

### Allowing nested event loops

Some LlamaIndex components run asynchronous code internally. Jupyter notebooks already run inside their own asyncio event loop, and by default Python does not allow one event loop to be started inside another. `nest_asyncio.apply()` patches this restriction so that LlamaIndex's internal asynchronous calls can run correctly inside a notebook environment without raising an error.


In [ ]:
import nest_asyncio

# Allow nested asyncio event loops in Jupyter
nest_asyncio.apply()

### Assembling the query engine

`RetrieverQueryEngine.from_args` takes the Auto Merging Retriever built earlier and wraps it into a complete query engine. Since the language model was already configured globally through `Settings.llm`, it does not need to be passed in again here.


In [ ]:
# Build the query engine using the Auto-Merging Retriever
query_engine = RetrieverQueryEngine.from_args(
    retriever
)

## Section 9. Asking a question

With the full pipeline assembled, a single question can now be sent through it end to end. Internally, the query engine performs the following sequence of steps.

1. Convert the question into an embedding using the configured embedding model.
2. Search the vector index for the most similar leaf nodes.
3. Check whether enough of the retrieved leaf nodes share a common parent, and merge them into that parent node when appropriate.
4. Send the resulting context, whether it consists of individual leaf nodes, merged parent nodes, or a mix of both, to the language model.
5. Generate and return the final natural language answer.


In [ ]:
# Ask a question
response = query_engine.query(
    "What are the different Chain of Thought prompting?"
)

# Display the generated answer
print(response)

## Section 10. The complete Auto Merging Retriever pipeline

The diagram below summarizes every step performed above, from the raw source document all the way through to the final generated answer, in the order those steps actually happen.

```text
Documents
     |
     v
HierarchicalNodeParser
     |
     v
Parent Nodes (1024)
     |
     v
Child Nodes (512)
     |
     v
Leaf Nodes (128)
     |
     v
VectorStoreIndex
     |
     v
Similarity Search
     |
     v
AutoMergingRetriever
(Merges nearby leaf nodes into parent nodes)
     |
     v
RetrieverQueryEngine
     |
     v
LLM
     |
     v
Final Answer
```

The step that distinguishes this pipeline from a typical flat chunking pipeline is the `AutoMergingRetriever` box. Every step before it is standard hierarchical parsing and vector search, but this is the point where retrieved leaf nodes are inspected and, when appropriate, silently upgraded into their larger parent node before the language model ever sees them.


## Conclusion

This notebook built a three level hierarchy out of a single source document, parent chunks of roughly 1024 tokens, child chunks of roughly 512 tokens, and leaf chunks of roughly 128 tokens, and indexed only the smallest leaf level chunks for similarity search, while keeping the entire hierarchy available in storage.

The Auto Merging Retriever then layered a simple but effective rule on top of ordinary similarity search. Search narrowly using the small, precise leaf chunks, but whenever several of the retrieved leaf chunks turn out to share a common parent, treat that as a signal that the whole parent region is relevant, and hand the language model that larger, more coherent parent chunk instead of several scattered leaf fragments. This is conceptually similar to the Sentence Window technique covered elsewhere in this series, in that both approaches search narrow and then expand context after retrieval, but Auto Merging Retrieval expands based on a tree structured hierarchy of chunk sizes rather than a fixed window of neighboring sentences.

The printed answer produced by the query engine, along with any merge related messages printed during retrieval thanks to `verbose=True`, shows concretely whether the automatic merging behavior triggered for this particular question and this particular source document. When it does trigger, the language model receives a single larger, more coherent piece of context instead of several smaller disconnected fragments, which can lead to a more complete and better grounded final answer.

Some natural next steps for extending this notebook include experimenting with different chunk sizes at each level of the hierarchy, adjusting `similarity_top_k` to see how it affects how often merging actually happens, adding the `SentenceTransformerRerank` postprocessor that was imported earlier to rerank results before merging, and comparing Auto Merging Retrieval against both naive flat chunking and Sentence Window Retrieval on the same set of questions to see which strategy performs best for a given kind of source document.
